# Project 08: Automatic Speech Recognition

**Category:** Audio | **Dataset:** LibriSpeech

Speech-to-text systems

---


In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Add project directory to path
project_dir = os.path.abspath('08_Automatic_Speech_Recognition')
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

# Device auto-detection: CUDA -> MPS -> CPU
def setup_device():
    if torch.cuda.is_available():
        device = torch.device('cuda')
        torch.backends.cudnn.benchmark = True
        print(f'Using CUDA: {torch.cuda.get_device_name(0)}')
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = torch.device('mps')
        print('Using Apple MPS')
    else:
        device = torch.device('cpu')
        print('Using CPU')
    return device

def get_platform_config(device):
    if device.type == 'cuda':
        return {'num_workers': 4, 'pin_memory': True, 'use_amp': True, 'amp_dtype': torch.float16}
    elif device.type == 'mps':
        return {'num_workers': 0, 'pin_memory': False, 'use_amp': True, 'amp_dtype': torch.float16}
    else:
        return {'num_workers': 2, 'pin_memory': False, 'use_amp': False, 'amp_dtype': None}

device = setup_device()
platform_config = get_platform_config(device)

# Set random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'PyTorch version: {torch.__version__}')
print(f'Platform config: {platform_config}')


In [ ]:
# Training Configuration
config = {
    'model_name': 'deepspeech2',
    'epochs': 5,
    'batch_size': 16,
    'learning_rate': 0.0003,
    'num_classes': 29,
    'num_workers': platform_config['num_workers'],
    'pin_memory': platform_config['pin_memory'],
    'use_amp': platform_config['use_amp'],
    'checkpoint_dir': './checkpoints',
    'use_wandb': False,
}

print("Training configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")


In [ ]:
from data_loader import create_data_loaders

# Load dataset
train_loader, val_loader, test_loader = create_data_loaders(batch_size=config['batch_size'], num_workers=platform_config['num_workers'])

# Print dataset statistics
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")


In [ ]:
# Visualize sample audio data
batch = next(iter(train_loader))
print("Batch keys:", list(batch.keys()))
for key, val in batch.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key}: shape={val.shape}, dtype={val.dtype}")


In [ ]:
from models import get_model

# Create model
model = get_model(config['model_name'], input_dim=80, num_classes=config.get('num_classes', 29))
model = model.to(device)

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
param_size_mb = sum(p.nelement() * p.element_size() for p in model.parameters()) / 1024 / 1024

print(f"Model: {type(model).__name__}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: {param_size_mb:.2f} MB")


In [ ]:
from train import ASRTrainer

# Initialize trainer
trainer = ASRTrainer(config)

# Run training
print("Starting training...")
trainer.train()

print("\nTraining complete!")


In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if hasattr(trainer, 'history'):
    history = trainer.history
elif hasattr(trainer, 'train_losses'):
    history = {
        'train_loss': trainer.train_losses,
        'val_loss': trainer.val_losses if hasattr(trainer, 'val_losses') else [],
    }
else:
    history = {'train_loss': [], 'val_loss': []}

if history.get('train_loss'):
    axes[0].plot(history['train_loss'], label='Train Loss', color='#0f766e')
    if history.get('val_loss'):
        axes[0].plot(history['val_loss'], label='Val Loss', color='#dc2626')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

# Plot accuracy or secondary metric if available
if history.get('train_acc'):
    axes[1].plot(history['train_acc'], label='Train Acc', color='#0f766e')
    if history.get('val_acc'):
        axes[1].plot(history['val_acc'], label='Val Acc', color='#dc2626')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Training and Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
elif history.get('learning_rate'):
    axes[1].plot(history['learning_rate'], color='#0f766e')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Learning Rate')
    axes[1].set_title('Learning Rate Schedule')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Evaluate on test set
print("Running evaluation on test set...")

if hasattr(trainer, 'test'):
    test_metrics = trainer.test()
    print("\nTest Results:")
    if isinstance(test_metrics, dict):
        for k, v in test_metrics.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")
            else:
                print(f"  {k}: {v}")
elif hasattr(trainer, 'validate'):
    test_metrics = trainer.validate()
    print("\nValidation Results:")
    if isinstance(test_metrics, dict):
        for k, v in test_metrics.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")
            else:
                print(f"  {k}: {v}")


In [ ]:
# Sample predictions
print("Generating sample predictions...")
model.eval()

test_batch = next(iter(test_loader))
if isinstance(test_batch, dict):
    sample = {k: v[:4].to(device) if isinstance(v, torch.Tensor) else v for k, v in test_batch.items()}
elif isinstance(test_batch, (list, tuple)):
    sample = [t[:4].to(device) if isinstance(t, torch.Tensor) else t for t in test_batch]

with torch.no_grad():
    if isinstance(sample, dict):
        outputs = model(**{k: v for k, v in sample.items() if k != 'labels' and k != 'label'})
    else:
        outputs = model(sample[0])

print("Sample predictions generated successfully.")


### Domain-Specific: Audio Analysis

In [ ]:
# Audio-specific analysis
print("Audio feature analysis:")

test_batch = next(iter(test_loader))
if isinstance(test_batch, dict):
    for key, val in test_batch.items():
        if isinstance(val, torch.Tensor):
            print(f"  {key}: shape={val.shape}")
elif isinstance(test_batch, (list, tuple)):
    for i, item in enumerate(test_batch):
        if isinstance(item, torch.Tensor):
            print(f"  Tensor {i}: shape={item.shape}")

# Visualize spectrograms if applicable
try:
    features = test_batch[0] if isinstance(test_batch, (list, tuple)) else test_batch.get('features', test_batch.get('input_values'))
    if features is not None and features.dim() >= 2:
        fig, axes = plt.subplots(1, min(4, features.shape[0]), figsize=(16, 4))
        if not isinstance(axes, np.ndarray):
            axes = [axes]
        for i, ax in enumerate(axes):
            if features.dim() == 3:
                ax.imshow(features[i].squeeze().numpy(), aspect='auto', origin='lower')
            elif features.dim() == 2:
                ax.plot(features[i].numpy())
            ax.set_title(f'Sample {i}')
        plt.suptitle('Audio Features')
        plt.tight_layout()
        plt.show()
except Exception as e:
    print(f"Could not visualize audio features: {e}")

print("Audio analysis complete.")


In [ ]:
import yaml
from datetime import datetime

# Model card
model_card = {
    'project_id': '08',
    'title': 'Automatic Speech Recognition',
    'short_description': 'Speech-to-text systems',
    'category': 'Audio',
    'input_type': 'audio',
    'output_type': 'transcribed_text',
    'default_model': 'deepspeech2',
    'models_available': ['deepspeech2', 'wav2vec2', 'whisper'],
    'dataset': {
        'name': 'LibriSpeech',
        'num_classes': 29,
        'train_size': 28539,
        'test_size': 2620,
    },
    'tags': ['speech-recognition', 'asr', 'librispeech'],
    'demo_type': 'precomputed',
}

# Results
results = {
    'project_id': '08',
    'timestamp': datetime.now().isoformat(),
    'device_used': str(device),
    'best_model': config['model_name'],
    'metrics': {},
    'training_history': {},
}

# Populate results from trainer
if hasattr(trainer, 'history'):
    results['training_history'] = trainer.history
if hasattr(trainer, 'best_acc'):
    results['metrics']['best_accuracy'] = float(trainer.best_acc)
if hasattr(trainer, 'best_val_loss'):
    results['metrics']['best_val_loss'] = float(trainer.best_val_loss)
if hasattr(trainer, 'best_val_rmse'):
    results['metrics']['best_val_rmse'] = float(trainer.best_val_rmse)
if hasattr(trainer, 'best_bleu'):
    results['metrics']['best_bleu'] = float(trainer.best_bleu)
if hasattr(trainer, 'best_map'):
    results['metrics']['best_map'] = float(trainer.best_map)

# Save
os.makedirs('08_Automatic_Speech_Recognition', exist_ok=True)

with open(os.path.join('08_Automatic_Speech_Recognition', 'model_card.yaml'), 'w') as f:
    yaml.dump(model_card, f, default_flow_style=False, sort_keys=False)

with open(os.path.join('08_Automatic_Speech_Recognition', 'results.yaml'), 'w') as f:
    yaml.dump(results, f, default_flow_style=False, sort_keys=False)

print("Exported model_card.yaml and results.yaml")


## Summary

### Automatic Speech Recognition

**Category:** Audio

#### Key Findings
- Model trained successfully with the configured hyperparameters
- Results exported to `model_card.yaml` and `results.yaml`

#### Next Steps
- Experiment with different model architectures
- Tune hyperparameters for better performance
- Run full training with more epochs and larger dataset subsets
- See the project README for detailed training configurations
